In [1]:
import nltk
nltk.download('brown')
from nltk.corpus import brown

[nltk_data] Downloading package brown to C:\Users\MSI
[nltk_data]     GF66\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


### Data Preparation

In [2]:
sentences = brown.sents()

sentences = [[word.lower() for word in sent] for sent in sentences]

print(len(sentences))
print(sentences[:5])

57340
[['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place', '.'], ['the', 'jury', 'further', 'said', 'in', 'term-end', 'presentments', 'that', 'the', 'city', 'executive', 'committee', ',', 'which', 'had', 'over-all', 'charge', 'of', 'the', 'election', ',', '``', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'city', 'of', 'atlanta', "''", 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', '.'], ['the', 'september-october', 'term', 'jury', 'had', 'been', 'charged', 'by', 'fulton', 'superior', 'court', 'judge', 'durwood', 'pye', 'to', 'investigate', 'reports', 'of', 'possible', '``', 'irregularities', "''", 'in', 'the', 'hard-fought', 'primary', 'which', 'was', 'won', 'by', 'mayor-nominate', 'ivan', 'allen', 'jr.', '.'], ['``', 'only', 'a', 'relative', 'handful', 'of', 'such'

In [3]:
sentences_clean = [
    [word for word in sent if word.isalpha()]
    for sent in sentences
]

print(sentences_clean[:5])

[['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place'], ['the', 'jury', 'further', 'said', 'in', 'presentments', 'that', 'the', 'city', 'executive', 'committee', 'which', 'had', 'charge', 'of', 'the', 'election', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'city', 'of', 'atlanta', 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted'], ['the', 'term', 'jury', 'had', 'been', 'charged', 'by', 'fulton', 'superior', 'court', 'judge', 'durwood', 'pye', 'to', 'investigate', 'reports', 'of', 'possible', 'irregularities', 'in', 'the', 'primary', 'which', 'was', 'won', 'by', 'ivan', 'allen'], ['only', 'a', 'relative', 'handful', 'of', 'such', 'reports', 'was', 'received', 'the', 'jury', 'said', 'considering', 'the', 'widespread', 'interest', 'in', 'the', 'election', 'the', 'number', 'of', 'voters', 'and', 'th

In [4]:
additional_sentences = [
    # Group 1: Morphologically Related Words
    ["the", "teacher", "is", "teaching", "students", "about", "grammar"],
    ["she", "teaches", "mathematics", "at", "the", "university"],
    ["the", "teacher", "prepared", "teaching", "materials", "yesterday"],
    ["many", "teachers", "attend", "teaching", "conferences", "annually"],
    ["effective", "teaching", "requires", "good", "communication", "skills"],

    # Group 2: Rare Morphological Variant
    ["the", "unteachable", "student", "refused", "to", "learn"],

    # Group 3: Compound and Derived Words
    ["computational", "linguistics", "combines", "computer", "science", "and", "language"],
    ["the", "computation", "took", "several", "hours", "to", "complete"],
    ["we", "computed", "the", "results", "using", "advanced", "algorithms"],
    ["modern", "computers", "can", "compute", "complex", "calculations", "quickly"],

    # Group 4: Rare Compound
    ["the", "recomputation", "was", "necessary", "after", "finding", "errors"],

    # Group 5: Another Morphological Family
    ["natural", "language", "processing", "is", "fascinating"],
    ["the", "nature", "of", "language", "is", "complex"],
    ["naturally", "occurring", "patterns", "in", "text", "are", "important"],

    # Group 6: Rare Morphological Variant
    ["the", "unnaturalness", "of", "the", "translation", "was", "obvious"]
]


### GloVe model

In [5]:
!pip install mittens

In [6]:
from mittens import GloVe

In [7]:
from collections import Counter, defaultdict
word_counts = Counter(w for sent in sentences_clean for w in sent)

In [8]:
vocabulary = [word for word, count in word_counts.items() if count > 5]

In [9]:
print(list(word_counts.items())[:10])

[('the', 69971), ('fulton', 17), ('county', 155), ('grand', 48), ('jury', 67), ('said', 1961), ('friday', 60), ('an', 3740), ('investigation', 51), ('of', 36412)]


In [10]:
additional_words = [word for sent in additional_sentences for word in sent]

vocabulary.extend([word for word in additional_words if word not in set(vocabulary)])

In [11]:
word_index = {word: i for i, word in enumerate(vocabulary)}
index_word = {i: word for word, i in word_index.items()}

In [12]:
print(vocabulary[:10])
print("Vocabulary size: ", len(vocabulary))
print(list(word_index.items())[:10])
print(list(index_word.items())[:10])

['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of']
Vocabulary size:  11749
[('the', 0), ('fulton', 1), ('county', 2), ('grand', 3), ('jury', 4), ('said', 5), ('friday', 6), ('an', 7), ('investigation', 8), ('of', 9)]
[(0, 'the'), (1, 'fulton'), (2, 'county'), (3, 'grand'), (4, 'jury'), (5, 'said'), (6, 'friday'), (7, 'an'), (8, 'investigation'), (9, 'of')]


In [13]:
import numpy as np
window_size = 5
co_occurrences = defaultdict(float)

for sent in sentences:
    for i, word in enumerate(sent):
        if word not in word_index:
            continue

        word_i = word_index[word]

        start = max(0, i - window_size)
        end = min(len(sent), i + window_size + 1)

        for j in range(start, end):
            if i == j:
                continue

            word2 = sent[j]
            if word2 not in word_index:
                continue

            word_j = word_index[word2]
            distance = abs(i-j)

            co_occurrences[(word_i, word_j)] += 1 / distance

vocab_size = len(vocabulary)
co_occurrence_matrix = np.zeros((vocab_size, vocab_size))

for (i, j), value in co_occurrences.items():
    co_occurrence_matrix[i, j] = value

print(co_occurrence_matrix[:5])

[[7.96240000e+03 8.06666667e+00 4.93833333e+01 ... 8.66666667e-01
  0.00000000e+00 0.00000000e+00]
 [8.06666667e+00 0.00000000e+00 6.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.93833333e+01 6.00000000e+00 1.06666667e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.59833333e+01 5.00000000e-01 1.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.63833333e+01 7.33333333e-01 5.00000000e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]]


### Model Training

In [14]:
import pickle

glove = GloVe(n=100,
              max_iter=10,
              learning_rate=0.05)
co_occurrence_matrix = co_occurrence_matrix.astype(np.float32)
embeddings = glove.fit(co_occurrence_matrix)

model = {'embeddings': embeddings,
         'word_index': word_index,
         'index_word': index_word}

with open('glove_model.pkl', 'wb') as f:
    pickle.dump(model, f)


Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor



Iteration 10: loss: 24470.3125

In [15]:
print("Embedding matrix shape:", embeddings.shape)

print(embeddings[word_index["word"]][:10])

Embedding matrix shape: (11749, 100)
[-0.01101279 -0.0618493   0.31333882 -0.26729205 -0.17061043 -0.20666693
 -0.3644173   0.00247489  0.04141672 -0.18683909]


### Inference

In [16]:
with open('glove_model.pkl', 'rb') as f:
    model = pickle.load(f)

embeddings = model['embeddings']
word_index = model['word_index']
index_word = model['index_word']

In [17]:
def get_vector(word):
    if word in word_index:
        return embeddings[word_index[word]]
    else:
        return None

print(get_vector("word")[:10])

[-0.01101279 -0.0618493   0.31333882 -0.26729205 -0.17061043 -0.20666693
 -0.3644173   0.00247489  0.04141672 -0.18683909]


In [18]:
from numpy.linalg import norm

def cosine_similarity(word1, word2):
    vector1 = get_vector(word1)
    vector2 = get_vector(word2)
    if word1 is None and word2 is None:
        return None
    return np.dot(vector1, vector2) / (norm(vector1) * norm(vector2))

print("Similarity school word and university", cosine_similarity("school", "university"))

Similarity school word and university 0.93317384


In [19]:
def most_similar(word, top_k=5):
    if word not in word_index:
        return []

    vector = get_vector(word)
    norms = norm(embeddings, axis=1) * norm(vector)
    similarities = embeddings.dot(vector) / norms

    best = np.argsort(similarities)[::-1][1:top_k+1]

    return [(index_word[i], similarities[i]) for i in best]

print(most_similar("university"))

[('center', 0.98929757), ('land', 0.98772067), ('top', 0.98731357), ('english', 0.98713666), ('level', 0.98705125)]


## Models Comparison

### OOV Words

In [20]:
from gensim.models import Word2Vec, FastText

w2v_model = Word2Vec.load("word2vec.model")

ft_model = FastText.load("fasttext.model")

In [21]:
OOV_words = ["teachable", "unteacher", "supercomputer", "miscomputation", "unnaturally"]

for word in OOV_words:
    if word in w2v_model.wv:
        w2v_vector = w2v_model.wv[word]
        print(f"{word}: FOUND, shape = {w2v_vector.shape}")
        print(w2v_vector[:20])
    else:
        print(f"{word}: NOT FOUND (OOV)")


teachable: NOT FOUND (OOV)
unteacher: NOT FOUND (OOV)
supercomputer: NOT FOUND (OOV)
miscomputation: NOT FOUND (OOV)
unnaturally: FOUND, shape = (100,)
[ 0.09243209  0.10831335  0.09878806  0.02158715  0.01840752 -0.21074852
  0.10815752  0.16726805 -0.15629824 -0.06127942 -0.17316745 -0.16178276
 -0.00849607  0.08065136  0.10004328 -0.03557203 -0.14236635 -0.03855184
  0.07609712 -0.2931619 ]


In [22]:
for word in OOV_words:
    ft_vector = ft_model.wv[word]
    print(ft_vector.shape)
    print(f"{word}", ft_vector[:10])

(100,)
teachable [-0.20627715  0.21301503 -0.26031998 -0.51886934  0.3548757   0.1640053
  0.09596927  0.76987135  0.01443066 -0.22707275]
(100,)
unteacher [ 0.12679751 -0.17822953 -0.04382741 -0.11309986  0.45280486 -0.18594116
  0.13141224  0.5613678  -0.53223914 -0.23106441]
(100,)
supercomputer [ 0.14826566  0.001673    0.02298766 -0.22285356  0.15738691 -0.03291582
  0.09838043  0.32132804 -0.20684262 -0.18740721]
(100,)
miscomputation [-0.0323016   0.17304905 -0.06469271 -0.19970077  0.06776691  0.01055942
  0.3540716   0.17987329 -0.16240783 -0.59349054]
(100,)
unnaturally [-0.49514696 -0.13502608  0.25135782  0.01561346  0.20579422 -0.10073999
  0.00263909  0.18381196  0.11275619 -0.47002557]


In [23]:
for word in OOV_words:
    if word in word_index:
        print(f"{word}: FOUND → vector shape {embeddings[word_index[word]].shape}")
    else:
        print(f"{word}: NOT FOUND (OOV)")

teachable: NOT FOUND (OOV)
unteacher: NOT FOUND (OOV)
supercomputer: NOT FOUND (OOV)
miscomputation: NOT FOUND (OOV)
unnaturally: NOT FOUND (OOV)


### Rare Words

In [24]:
def cosine_similarity_vec(vec1, vec2):
    return np.dot(vec1, vec2) / (norm(vec1) * norm(vec2))


rare_words = ["unteachable", "recomputation", "unnaturalness"]
common_words = ["teacher", "computation", "natural"]

for word1, word2 in zip(rare_words, common_words):
    similarities = []

    if word1 in w2v_model.wv and word2 in w2v_model.wv:
        wv_sim = w2v_model.wv.similarity(word1, word2)
        similarities.append(wv_sim)
        print(f"Word2Vec similarity of {word1} and {word2}: {wv_sim}")
    else:
        wv_sim = -1
        print(f"Word2Vec similarity of {word1} and {word2}: OOV")

    ft_sim = ft_model.wv.similarity(word1, word2)
    similarities.append(ft_sim)
    print(f"FastText similarity of {word1} and {word2}: {ft_sim}")

    if word1 in word_index and word2 in word_index:
        v1 = embeddings[word_index[word1]]
        v2 = embeddings[word_index[word2]]
        glove_sim = cosine_similarity_vec(v1, v2)
        similarities.append(glove_sim)
        print(f"GloVe similarity of {word1} and {word2}: {glove_sim}")
    else:
        glove_sim = -1
        print(f"GloVe similarity of {word1} and {word2}: OOV")


    if max(similarities) == ft_sim:
        print("FastText shows stronger similarity")
    elif max(similarities) == wv_sim:
        print("Word2Vec shows stronger similarity")
    elif max(similarities) == glove_sim:
        print("GloVe shows stronger similarity")


Word2Vec similarity of unteachable and teacher: 0.49298295378685
FastText similarity of unteachable and teacher: 0.5979201793670654
GloVe similarity of unteachable and teacher: 0.034160029143095016
FastText shows stronger similarity
Word2Vec similarity of recomputation and computation: 0.6526901125907898
FastText similarity of recomputation and computation: 0.9588416218757629
GloVe similarity of recomputation and computation: -0.10283438861370087
FastText shows stronger similarity
Word2Vec similarity of unnaturalness and natural: 0.5403430461883545
FastText similarity of unnaturalness and natural: 0.7457697987556458
GloVe similarity of unnaturalness and natural: -0.6983910799026489
FastText shows stronger similarity


###  Morphological Relationships

In [25]:
morph_rel = ["teaching", "teach", "computation", "compute"]

for word in morph_rel:
    wv = w2v_model.wv.most_similar(word, topn=5)
    ft = ft_model.wv.most_similar(word, topn=5)
    glove = most_similar(word, top_k=5)

    print(f"Top 5 most similar words to '{word}':")
    print("Word2Vec: ", wv)
    print("FastText: ", ft)
    print("GloVe: ", glove)
    print(f"{'_'*70}")
    print()


Top 5 most similar words to 'teaching':
Word2Vec:  [('assimilation', 0.7218841910362244), ('marketable', 0.7090026140213013), ('elementary-school', 0.6986861824989319), ('compulsivity', 0.6984511613845825), ('fabrication', 0.6897518038749695)]
FastText:  [('teachings', 0.8875789642333984), ('aching', 0.8830503225326538), ('out-reaching', 0.8653694987297058), ('sky-reaching', 0.8644314408302307), ('beaching', 0.860158383846283)]
GloVe:  [('completion', 0.96859944), ('scientific', 0.96808577), ('pleasure', 0.96785706), ('weather', 0.9673441), ('interests', 0.9670599)]
______________________________________________________________________

Top 5 most similar words to 'teach':
Word2Vec:  [('translate', 0.7983649373054504), ('librarian', 0.7759448885917664), ('approve', 0.7646365761756897), ('recruit', 0.7585350275039673), ('disturb', 0.7575100660324097)]
FastText:  [('teacher', 0.8378256559371948), ('unteach', 0.8233897089958191), ("teachers'", 0.802670419216156), ('teaches', 0.80217802524

###  Morphological Analogies

In [26]:
positive_words = [["teaching", "computer"], ["naturally", "quick"]]
negative_words = ["teacher", "natural"]

for i in range(len(negative_words)):
    print(f"{positive_words[i][0]} : {positive_words[i][1]} = {negative_words[i]} : ? ")

    print("Word2Vec morphological analogies")
    test_w2v = w2v_model.wv.most_similar(positive=positive_words[i],
                                         negative=negative_words[i],
                                         topn=5)
    print(test_w2v)

    print("FastText morphological analogies")
    test_ft = ft_model.wv.most_similar(positive=positive_words[i],
                                       negative=negative_words[i],
                                       topn=5)
    print(test_ft)

    print("GloVe morphological analogies")
    vec = get_vector(positive_words[i][0]) - get_vector(negative_words[i]) + get_vector(positive_words[i][1])
    norms = norm(embeddings, axis=1) * norm(vec)
    similarities = embeddings.dot(vec) / norms
    best_idx = np.argsort(similarities)[::-1][:5]
    glove_results = [(index_word[idx], similarities[idx]) for idx in best_idx]
    print(glove_results)

    print(f"{'_'*70}\n")


teaching : computer = teacher : ? 
Word2Vec morphological analogies
[('lookup', 0.7337884306907654), ('manifold', 0.7054128050804138), ('ultraviolet', 0.6857830286026001), ('developmental', 0.6831170916557312), ('glottochronological', 0.6811620593070984)]
FastText morphological analogies
[('computing', 0.8557388782501221), ('sampling', 0.746216893196106), ('compiling', 0.7445888519287109), ('amplifying', 0.7397489547729492), ('multiplying', 0.7387749552726746)]
GloVe morphological analogies
[('occupational', 0.61260736), ('spatial', 0.5743987), ('storms', 0.57404655), ('vincent', 0.570429), ('architectural', 0.5672915)]
______________________________________________________________________

naturally : quick = natural : ? 
Word2Vec morphological analogies
[('careful', 0.5796281099319458), ('cognac', 0.5438370704650879), ('dubious', 0.5359185338020325), ('dumb', 0.5358986258506775), ('trig', 0.5355992317199707)]
FastText morphological analogies
[('quickly', 0.8022687435150146), ('quicki